In [1]:
import pandas as pd
import os
from google.colab import drive
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist
import numpy as np
#Se importan modelos de aprendizaje y tambien el de comparación preentrenado (Textblob)
!pip install textblob
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
df_train = '/content/drive/My Drive/Colab Notebooks/TP3_DIPLO/Processed/train_topicos.csv'

In [3]:
df_train = pd.read_csv(df_train)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Cargar tus datos (asumiendo que df_train es tu único archivo fuente)
# Reemplaza con la ruta real de tu archivo
df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/TP3_DIPLO/Processed/train_topicos.csv')
df['text_avanzado'] = df['text_avanzado'].fillna('')

# 2. Hacemos el split AHORA (antes de cualquier otra cosa)
# Esto garantiza que el TEST nunca se tocará durante el entrenamiento
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

# 3. Vectorización (Fit solo con el Train, Transform con ambos)
tfidf = TfidfVectorizer(max_features=None, ngram_range=(1,2))
X_train_vec = tfidf.fit_transform(df_train['text_avanzado'])
X_test_vec = tfidf.transform(df_test['text_avanzado'])

# 4. Entrenamiento
lr_model = LogisticRegression(solver='saga', n_jobs=-1, class_weight='balanced')
lr_model.fit(X_train_vec, df_train['target'])

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, df_train['target'])

# 5. Función de umbral (para la lógica de clase 2)
def get_predictions_with_threshold(model, X_data, threshold=0.6):
    probs = model.predict_proba(X_data)
    max_probs = probs.max(axis=1)
    predictions = model.predict(X_data)
    return np.where(max_probs < threshold, 2, predictions)

In [ ]:
import joblib

export_path = '/content/drive/My Drive/Colab Notebooks/TP3_DIPLO/Models/'

joblib.dump(tfidf, export_path + 'tfidf_vectorizer.joblib')
joblib.dump(lr_model, export_path + 'lr_model.joblib')
joblib.dump(nb_model, export_path + 'nb_model.joblib')

print(f"Modelos y vectorizador guardados exitosamente en: {export_path}")

Modelos y vectorizador guardados exitosamente en: /content/drive/My Drive/Colab Notebooks/TP3_DIPLO/Models/
